# 04 - Marketing & LTV

Customer lifetime value, value tiers, and a CAC:LTV view.

In [1]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from data_prep import clean_transactions, summarize
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize']=(10,5)
from features import build_rfm

In [2]:
# Load raw data (after placing online_retail_II.xlsx in ../data/raw/)
raw = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name=None)
df = pd.concat(raw.values(), ignore_index=True)
df = clean_transactions(df)
summarize(df)

{'rows': 805549,
 'customers': 5878,
 'invoices': 36969,
 'first_order': Timestamp('2009-12-01 07:45:00'),
 'last_order': Timestamp('2011-12-09 12:50:00'),
 'total_revenue': np.float64(17743429.18)}

## Revenue concentration (Pareto)

In [3]:
ltv = df.groupby('customer_id')['revenue'].sum().sort_values(ascending=False)
cum = ltv.cumsum()/ltv.sum()
top20 = cum.reset_index(drop=True).iloc[int(len(ltv)*0.2)]
print(f'Top 20% of customers drive {top20:.0%} of revenue')

Top 20% of customers drive 77% of revenue


## CAC : LTV by segment (simulated CAC)

In [4]:
rfm = build_rfm(df)
CAC = 8.0  # example blended CAC; replace with your real acquisition cost
seg = rfm.groupby('segment')['monetary'].mean().rename('avg_ltv').reset_index()
seg['cac'] = CAC
seg['ltv_to_cac'] = (seg['avg_ltv']/seg['cac']).round(1)
print(seg.sort_values('ltv_to_cac', ascending=False))

           segment      avg_ltv  cac  ltv_to_cac
1        Champions  8294.965354  8.0      1036.9
2            Loyal  3276.505428  8.0       409.6
4  New / Promising   998.946588  8.0       124.9
3  Needs Attention   940.119869  8.0       117.5
0          At Risk   438.031466  8.0        54.8


## Simulated A/B test on a retention email

In [5]:
from scipy import stats
np.random.seed(0)
control = np.random.binomial(1, 0.11, 4000)
variant = np.random.binomial(1, 0.135, 4000)
z, p = stats.ttest_ind(variant, control)
print(f'Control: {control.mean():.1%}  Variant: {variant.mean():.1%}  p={p:.4f}')
print('Lift:', f'{(variant.mean()-control.mean())/control.mean():.0%}')

Control: 11.6%  Variant: 12.7%  p=0.1604
Lift: 9%


> **Done.** Summarize findings in the README and screenshot the dashboard.